In [2]:
from datasets import load_from_disk, load_dataset

In [3]:
# Download dataset from hugging face
# d = load_dataset("Salesforce/wikitext", "wikitext-103-v1")
# d.save_to_disk("wikitext-103-v1")

In [103]:
%%time
data = load_from_disk("wikitext-103-v1")
data

CPU times: user 7.01 ms, sys: 49.2 ms, total: 56.2 ms
Wall time: 185 ms


DatasetDict({
    test: Dataset({
        features: ['text'],
        num_rows: 4358
    })
    train: Dataset({
        features: ['text'],
        num_rows: 1801350
    })
    validation: Dataset({
        features: ['text'],
        num_rows: 3760
    })
})

In [104]:
%%time
# Get a list of training tests
text_list = list(data["train"]["text"])

# Prune almost empty documents (<30 characters, arbitray threshold)
pruned_text_list = [t for t in text_list if len(t) > 100]\

len(pruned_text_list)

CPU times: user 4.71 s, sys: 93 ms, total: 4.8 s
Wall time: 4.8 s


749331

In [1]:
import sentencepiece
import io
from infoweight import InformationWeightTransformer
from vectorizers import TokenCooccurrenceVectorizer, NgramVectorizer, WassersteinVectorizer

import pickle

In [106]:
model_buffer = io.BytesIO()
sentencepiece.SentencePieceTrainer.Train(
    sentence_iterator=(x for x in pruned_text_list), model_writer=model_buffer, vocab_size=10000
)
tokenizer = sentencepiece.SentencePieceProcessor(model_proto=model_buffer.getvalue())
token_sequences = tokenizer.encode(pruned_text_list, out_type=str)

with open("tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)

I0000 00:00:1794272991.061286 3558424 sentencepiece_trainer.cc:149] Starts training with : 
trainer_spec {
  input_format: 
  model_prefix: 
  model_type: UNIGRAM
  vocab_size: 10000
  character_coverage: 0.9995
  input_sentence_size: 0
  shuffle_input_sentence: 1
  seed_sentencepiece_size: 1000000
  shrinking_factor: 0.75
  max_sentence_length: 4192
  num_threads: 16
  num_sub_iterations: 2
  max_sentencepiece_length: 16
  split_by_unicode_script: 1
  split_by_number: 1
  split_by_whitespace: 1
  split_digits: 0
  pretokenization_delimiter: 
  treat_whitespace_as_suffix: 0
  allow_whitespace_only_pieces: 0
  required_chars: 
  byte_fallback: 0
  auto_character_coverage: 0
  vocabulary_output_piece_score: 1
  seed_sentencepieces_file: 
  hard_vocab_limit: 1
  use_all_vocab: 0
  unk_id: 0
  bos_id: 1
  eos_id: 2
  pad_id: -1
  unk_piece: <unk>
  bos_piece: <s>
  eos_piece: </s>
  pad_piece: <pad>
  unk_surface:  ⁇ 
}
normalizer_spec {
  name: nmt_nfkc
  add_dummy_prefix: 1
  remove_extr

In [107]:
unique_tokens = list(set.union(*[set(s) for s in token_sequences]))
token_dictionary = {t:i for i,t in enumerate(unique_tokens)}

with open("tokenizer.pkl", "wb") as f:
    pickle.dump(token_dictionary, f)
len(unique_tokens)

11316

In [2]:
with open("token_dictionary.pkl", "rb") as f:
    token_dictionary = pickle.load(f)
with open("token_sequences.pkl", "rb") as f:
    token_sequences = pickle.load(f)

In [3]:
skipgram_vectorizer = TokenCooccurrenceVectorizer(token_dictionary=token_dictionary)
skipgrams = skipgram_vectorizer.fit_transform(token_sequences)
skipgrams

/Users/ryandewolfe33/Research/zero-parameter-embedder/.venv/lib/python3.14/site-packages/numba/core/utils.py:621: NumbaExperimentalFeatureWarning: First-class function type feature is experimental
  warnings.warn("First-class function type feature is experimental",


<Compressed Sparse Row sparse matrix of dtype 'float32'
	with 57859060 stored elements and shape (11316, 22632)>

In [9]:
from sklearn.preprocessing import normalize

sk = normalize(skipgrams, norm="l1", axis=1).astype("float32")
sk = sk.sqrt()

In [10]:
import scipy.sparse.linalg

u, s, v = scipy.sparse.linalg.svds(sk, k=129)

In [12]:
s.min()

np.float32(1.7841448)

In [109]:
%%time
# Reduce dimension of token_vectors using SVD with Hellinger Distance
from sklearn.preprocessing import normalize
from sklearn.decomposition import TruncatedSVD

# Row normalize token vectors into distributions, also decrease precision
skipgram_distributions = normalize(skipgrams, norm="l1", axis=1).astype("float32")

# Reduct dimenion using MDS, applied to sqrt of token_distributions
svd = TruncatedSVD(n_components=100, random_state=42)
token_vectors_skipgram_raw = svd.fit_transform(skipgram_distributions.sqrt())

with open("token_vectors_skipgram_raw.pkl", "wb") as f:
    pickle.dump(token_vectors_skipgram_raw, f)

token_vectors_skipgram_raw.shape

CPU times: user 9.06 s, sys: 159 ms, total: 9.22 s
Wall time: 9.37 s


(11316, 100)

In [110]:
%%time
iwt = InformationWeightTransformer()
token_vectors = iwt.fit_transform(skipgrams).tocsr()
token_vectors

CPU times: user 1.08 s, sys: 97.2 ms, total: 1.18 s
Wall time: 761 ms


<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 57859060 stored elements and shape (11316, 22632)>

In [111]:
%%time
# Reduce dimension of token_vectors using SVD with Hellinger Distance
from sklearn.preprocessing import normalize
from sklearn.decomposition import TruncatedSVD

# Row normalize token vectors into distributions, also decrease precision
token_distributions = normalize(token_vectors, norm="l1", axis=1).astype("float32")

# Reduct dimenion using MDS, applied to sqrt of token_distributions
svd = TruncatedSVD(n_components=100, random_state=42)
token_vectors_skipgram_iwt = svd.fit_transform(token_distributions.sqrt())

with open("token_vectors_skipgram_iwt.pkl", "wb") as f:
    pickle.dump(token_vectors_skipgram_iwt, f)

token_vectors_skipgram_iwt.shape

CPU times: user 9.05 s, sys: 110 ms, total: 9.16 s
Wall time: 9.22 s


(11316, 100)

In [112]:
%%time
counter = NgramVectorizer(token_dictionary=token_dictionary)
token_counts = counter.fit_transform(token_sequences)
token_counts

CPU times: user 1min 12s, sys: 21.2 s, total: 1min 33s
Wall time: 1min 41s


KeyboardInterrupt: 

In [14]:
%%time
wasserstein_raw_raw = WassersteinVectorizer(method="LOT_sinkhorn", random_state=42)
wasserstein_raw_raw.fit(token_counts, vectors=token_vectors_skipgram_raw)

with open("wasserstein_raw_raw.pkl", "wb") as f:
    pickle.dump(wasserstein_raw_raw, f)

CPU times: user 7.15 s, sys: 1.46 s, total: 8.61 s
Wall time: 5.13 s


In [15]:
%%time
wasserstein_iwt_raw = WassersteinVectorizer(method="LOT_sinkhorn", random_state=42)
wasserstein_iwt_raw.fit(token_counts, vectors=token_vectors_skipgram_iwt)

with open("wasserstein_iwt_raw.pkl", "wb") as f:
    pickle.dump(wasserstein_iwt_raw, f)

CPU times: user 4.87 s, sys: 1.59 s, total: 6.45 s
Wall time: 2.98 s


In [17]:
%%time
informationweight_transformer = InformationWeightTransformer()
iwt_token_counts = informationweight_transformer.fit_transform(token_counts).tocsr()

with open("infoweight_transformer.pkl", "wb") as f:
    pickle.dump(informationweight_transformer , f)

iwt_token_counts

CPU times: user 26.9 ms, sys: 4.27 ms, total: 31.2 ms
Wall time: 34 ms


<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 1544916 stored elements and shape (15290, 10150)>

In [18]:
%%time
wasserstein_raw_iwt = WassersteinVectorizer(method="LOT_sinkhorn", random_state=42)
wasserstein_raw_iwt.fit(iwt_token_counts, vectors=token_vectors_skipgram_raw)

with open("wasserstein_raw_iwt.pkl", "wb") as f:
    pickle.dump(wasserstein_raw_iwt, f)

CPU times: user 4.86 s, sys: 2.01 s, total: 6.87 s
Wall time: 2.76 s


In [19]:
%%time
wasserstein_iwt_iwt = WassersteinVectorizer(method="LOT_sinkhorn", random_state=42)
wasserstein_iwt_iwt.fit(iwt_token_counts, vectors=token_vectors_skipgram_iwt)

with open("wasserstein_iwt_iwt.pkl", "wb") as f:
    pickle.dump(wasserstein_iwt_iwt, f)

CPU times: user 4.72 s, sys: 680 ms, total: 5.4 s
Wall time: 2.71 s


# Test it works

Test the each model on a single small dataset. For the real run use a script

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from embedder import ZeroParameterEmbedder
from mteb.models import EncoderProtocol
from mteb.models.abs_encoder import AbsEncoder
from mteb.models.model_meta import ModelMeta
import pickle
import time

In [4]:
with open("tokenizer.pkl", "rb") as f:
    tokenizer = pickle.load(f)

with open("token_dictionary.pkl", "rb") as f:
    token_dictionary = pickle.load(f)

with open("token_vectors_skipgram_raw.pkl", "rb") as f:
    token_vectors_skipgram_raw = pickle.load(f)

with open("token_vectors_skipgram_iwt.pkl", "rb") as f:
    token_vectors_skipgram_iwt = pickle.load(f)

with open("token_vectors_skipgram.pkl", "rb") as f:
    token_vectors_skipgram = pickle.load(f)

with open("infoweight_transformer.pkl", "rb") as f:
    infoweight_transformer = pickle.load(f)

with open("wasserstein_raw_raw.pkl", "rb") as f:
    wasserstein_raw_raw = pickle.load(f)

with open("wasserstein_iwt_raw.pkl", "rb") as f:
    wasserstein_iwt_raw = pickle.load(f)

with open("wasserstein_raw_iwt.pkl", "rb") as f:
    wasserstein_raw_iwt = pickle.load(f)

with open("wasserstein_iwt_iwt.pkl", "rb") as f:
    wasserstein_iwt_iwt = pickle.load(f)

with open("wasserstein_w2v_raw.pkl", "rb") as f:
    wasserstein_w2v_raw = pickle.load(f)

with open("wasserstein_w2v_iwt.pkl", "rb") as f:
    wasserstein_w2v_iwt = pickle.load(f)

In [5]:
zpe1 = ZeroParameterEmbedder(
    "ZPE",
    "0.0.1",
    tokenizer,
    token_dictionary,
    token_vectors_skipgram_raw,
    None,
    wasserstein_raw_raw,
)
zpe1.mteb_model_meta = ModelMeta(
    name="uw/zpe_raw_raw",
    revision="0.0.1",
    languages=["eng-Latn"],
    n_parameters=None,
    n_embedding_parameters=None,
    loader= lambda **kwargs : None,
    max_tokens=None,
    embed_dim=128,
    release_date="2026-11-09",
    memory_usage_mb=None,
    open_weights=True,
    public_training_code="True",
    public_training_data="True",
    framework=[],
    similarity_fn_name="cosine",
    use_instructions=None,
    training_datasets=["wikitext"],
    license=None,
)

zpe2 = ZeroParameterEmbedder(
    "ZPE",
    "0.0.1",
    tokenizer,
    token_dictionary,
    token_vectors_skipgram_iwt,
    None,
    wasserstein_iwt_raw,
)
zpe2.mteb_model_meta = ModelMeta(
    name="uw/zpe_iwt_raw",
    revision="0.0.1",
    languages=["eng-Latn"],
    n_parameters=None,
    n_embedding_parameters=None,
    loader= lambda **kwargs : None,
    max_tokens=None,
    embed_dim=128,
    release_date="2026-11-09",
    memory_usage_mb=None,
    open_weights=True,
    public_training_code="True",
    public_training_data="True",
    framework=[],
    similarity_fn_name="cosine",
    use_instructions=None,
    training_datasets=["wikitext"],
    license=None,
)

zpe3 = ZeroParameterEmbedder(
    "ZPE",
    "0.0.1",
    tokenizer,
    token_dictionary,
    token_vectors_skipgram_raw,
    infoweight_transformer,
    wasserstein_raw_iwt,
)
zpe3.mteb_model_meta = ModelMeta(
    name="uw/zpe_raw_iwt",
    revision="0.0.1",
    languages=["eng-Latn"],
    n_parameters=None,
    n_embedding_parameters=None,
    loader= lambda **kwargs : None,
    max_tokens=None,
    embed_dim=128,
    release_date="2026-11-09",
    memory_usage_mb=None,
    open_weights=True,
    public_training_code="True",
    public_training_data="True",
    framework=[],
    similarity_fn_name="cosine",
    use_instructions=None,
    training_datasets=["wikitext"],
    license=None,
)

zpe4 = ZeroParameterEmbedder(
    "ZPE",
    "0.0.1",
    tokenizer,
    token_dictionary,
    token_vectors_skipgram_iwt,
    infoweight_transformer,
    wasserstein_iwt_iwt,
)
zpe4.mteb_model_meta = ModelMeta(
    name="uw/zpe_iwt_iwt",
    revision="0.0.1",
    languages=["eng-Latn"],
    n_parameters=None,
    n_embedding_parameters=None,
    loader= lambda **kwargs : None,
    max_tokens=None,
    embed_dim=128,
    release_date="2026-11-09",
    memory_usage_mb=None,
    open_weights=True,
    public_training_code="True",
    public_training_data="True",
    framework=[],
    similarity_fn_name="cosine",
    use_instructions=None,
    training_datasets=["wikitext"],
    license=None,
)

zpe5 = ZeroParameterEmbedder(
    "ZPE",
    "0.0.1",
    tokenizer,
    token_dictionary,
    token_vectors_skipgram,
    None,
    wasserstein_w2v_raw,
)
zpe5.mteb_model_meta = ModelMeta(
    name="uw/zpe_w2v_raw",
    revision="0.0.1",
    languages=["eng-Latn"],
    n_parameters=None,
    n_embedding_parameters=None,
    loader= lambda **kwargs : None,
    max_tokens=None,
    embed_dim=128,
    release_date="2026-11-09",
    memory_usage_mb=None,
    open_weights=True,
    public_training_code="True",
    public_training_data="True",
    framework=[],
    similarity_fn_name="cosine",
    use_instructions=None,
    training_datasets=["wikitext"],
    license=None,
)

zpe6 = ZeroParameterEmbedder(
    "ZPE",
    "0.0.1",
    tokenizer,
    token_dictionary,
    token_vectors_skipgram,
    infoweight_transformer,
    wasserstein_w2v_iwt,
)
zpe6.mteb_model_meta = ModelMeta(
    name="uw/zpe_w2v_iwt",
    revision="0.0.1",
    languages=["eng-Latn"],
    n_parameters=None,
    n_embedding_parameters=None,
    loader= lambda **kwargs : None,
    max_tokens=None,
    embed_dim=128,
    release_date="2026-11-09",
    memory_usage_mb=None,
    open_weights=True,
    public_training_code="True",
    public_training_data="True",
    framework=[],
    similarity_fn_name="cosine",
    use_instructions=None,
    training_datasets=["wikitext"],
    license=None,
)

In [6]:
all(isinstance(zpe, EncoderProtocol) for zpe in [zpe1, zpe2, zpe3, zpe4, zpe5, zpe6])

True

In [7]:
import mteb
tasks = mteb.get_tasks(tasks=["ArXivHierarchicalClusteringS2S"])

In [18]:
# Get the full English benchmark
benchmark = mteb.get_benchmark("MTEB(eng, v2)")

# Filter to only retrieval tasks
filtered_tasks = [
    task for task in benchmark.tasks if task.metadata.type in [
        "Classification",
        "Clustering",
        "Reranking",
        "Summarization",
    ]]
print(f"Found {len(filtered_tasks)} tasks")

Found 19 tasks


In [19]:
from dotenv import load_dotenv
import os
load_dotenv()
os.environ["HF_TOKEN"] = os.getenv("HF-KEY")

custom_cache = mteb.cache.ResultCache(cache_path="test_results_cache")
for zpe in [zpe1, zpe2, zpe3, zpe4, zpe5, zpe6]:
    print(zpe.model_name, zpe.revision)
    time.sleep(0.2)
    results = mteb.evaluate(
        zpe, 
        tasks=filtered_tasks,
        cache=custom_cache,
    )
    df = results.to_dataframe()
    print(df)
    time.sleep(1)
    print()
    print()

ZPE 0.0.1


Evaluating tasks:   0%|          | 0/19 [00:00<?, ?it/s]

test.jsonl.gz: reconstructing file:   0%|          |  0.00B /  270MB            

test.jsonl.gz: downloading bytes:           |  0.00B            

Generating test split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/732723 [00:00<?, ? examples/s]

... (791 outputs left out)

Generating train split:   0%|          | 0/207455003 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/97006943 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/97006943 [00:00<?, ? examples/s]

KeyboardInterrupt: 

In [1]:
import bz2
import pickle

In [5]:
with open("token_sequences.pkl", "rb") as f:
    x = pickle.load(f)
x

[['▁Sen',
  'j',
  'ō',
  '▁no',
  '▁V',
  'alk',
  'y',
  'ria',
  '▁3',
  '▁:',
  '▁',
  '<',
  'unk',
  '>',
  '▁Chronic',
  'les',
  '▁(',
  '▁Japanese',
  '▁:',
  '▁',
  '戦場のヴァルキュリア',
  '3',
  '▁,',
  '▁lit',
  '▁.',
  '▁V',
  'alk',
  'y',
  'ria',
  '▁of',
  '▁the',
  '▁Battle',
  'field',
  '▁3',
  '▁)',
  '▁,',
  '▁commonly',
  '▁referred',
  '▁to',
  '▁as',
  '▁V',
  'alk',
  'y',
  'ria',
  '▁Chronic',
  'les',
  '▁III',
  '▁outside',
  '▁Japan',
  '▁,',
  '▁is',
  '▁a',
  '▁tactical',
  '▁role',
  '▁@-@',
  '▁playing',
  '▁video',
  '▁game',
  '▁developed',
  '▁by',
  '▁Sega',
  '▁and',
  '▁Media',
  '.',
  'V',
  'ision',
  '▁for',
  '▁the',
  '▁PlayStation',
  '▁Port',
  'able',
  '▁.',
  '▁Re',
  'leased',
  '▁in',
  '▁January',
  '▁2011',
  '▁in',
  '▁Japan',
  '▁,',
  '▁it',
  '▁is',
  '▁the',
  '▁third',
  '▁game',
  '▁in',
  '▁the',
  '▁V',
  'alk',
  'y',
  'ria',
  '▁series',
  '▁.',
  '▁Em',
  'ploy',
  'ing',
  '▁the',
  '▁same',
  '▁fusion',
  '▁of',
  '▁tactica

In [6]:
with bz2.open("token_sequences.pkl.bz2", "wb") as f:
    pickle.dump(x, f, protocol=pickle.HIGHEST_PROTOCOL)

In [8]:
files = (
    "infoweight_transformer.pkl",
    "token_dictionary.pkl",
    "token_vectors_skipgram_iwt.pkl",
    "token_vectors_skipgram_raw.pkl",
    "token_vectors_skipgram.pkl",
    "wasserstein_raw_raw.pkl",
    "wasserstein_raw_iwt.pkl",
    "wasserstein_iwt_raw.pkl",
    "wasserstein_iwt_iwt.pkl",
    "wasserstein_w2v_raw.pkl",
    "wasserstein_w2v_iwt.pkl",
)

for file in files:
    print(file)
    with open(file, "rb") as f:
        x = pickle.load(f)
    with bz2.open(file+".bz2", "wb") as f:
        pickle.dump(x, f, protocol=pickle.HIGHEST_PROTOCOL)

infoweight_transformer.pkl
token_dictionary.pkl
token_vectors_skipgram_iwt.pkl
token_vectors_skipgram_raw.pkl
token_vectors_skipgram.pkl
wasserstein_raw_raw.pkl
wasserstein_raw_iwt.pkl
wasserstein_iwt_raw.pkl
wasserstein_iwt_iwt.pkl
wasserstein_w2v_raw.pkl
wasserstein_w2v_iwt.pkl
